## Cell 9 — Live Camera Test (Google Colab)

بيصور من كاميرا المتصفح مباشرةً لمدة **35 ثانية** وبعدين بيبعت notification بالنتيجة.

> لو عايز كاميرا الموبايل — غيّر `USE_MOBILE_CAM = True` وحط الـ IP بتاعك.

In [ ]:
# ============================================================
# إعدادات
# ============================================================
TEST_DURATION   = 35       # ثانية
CHEAT_THRESHOLD = 0.5      # لو أكتر من 50% من الـ frames غش → CHEATING
CAPTURE_EVERY   = 2        # صور frame كل كام ثانية
USE_MOBILE_CAM  = False    # True لو عايز كاميرا الموبايل بدل اللاب
MOBILE_CAM_URL  = 'http://192.168.1.5:8080/video'   # ← غيّر الـ IP بتاعك

IMG_SIZE        = (224, 224)
CLASS_NAMES     = ['cheating', 'normal']   # ترتيب أبجدي

# ============================================================
# Imports
# ============================================================
import numpy as np
import time
import base64
import cv2
from IPython.display import display, Javascript, HTML, clear_output
from google.colab.output import eval_js
from tensorflow import keras
import tensorflow as tf

# ============================================================
# Predict على frame واحدة
# ============================================================
def predict_frame(frame, loaded_model):
    img = cv2.resize(frame, IMG_SIZE)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = img.astype(np.float32) / 255.0
    img = np.expand_dims(img, axis=0)
    prob = loaded_model.predict(img, verbose=0)[0][0]
    label = CLASS_NAMES[int(prob > 0.5)]
    return label, float(prob)

# ============================================================
# تحويل base64 → OpenCV frame
# ============================================================
def base64_to_frame(b64_str):
    # شيل الـ header لو موجود
    if ',' in b64_str:
        b64_str = b64_str.split(',')[1]
    img_bytes = base64.b64decode(b64_str)
    np_arr   = np.frombuffer(img_bytes, np.uint8)
    frame    = cv2.imdecode(np_arr, cv2.IMREAD_COLOR)
    return frame

# ============================================================
# Notification
# ============================================================
def send_notification(result, cheat_ratio):
    if result == 'CHEATING':
        color = '#ff4444'
        icon  = '⚠️'
        msg   = f'تم اكتشاف غش! ({cheat_ratio:.0%} من الوقت)'
    else:
        color = '#44bb44'
        icon  = '✅'
        msg   = f'لا يوجد غش ({1-cheat_ratio:.0%} طبيعي)'

    # رسالة HTML على الشاشة
    display(HTML(f"""
    <div style="
        background:{color}; color:white;
        padding:20px; border-radius:12px;
        font-size:22px; font-weight:bold;
        text-align:center; margin:10px;
    ">
        {icon} &nbsp; {msg}
    </div>
    """))

    # صوت تنبيه من المتصفح
    if result == 'CHEATING':
        display(Javascript("""
            var ctx = new AudioContext();
            for (var i = 0; i < 3; i++) {
                var osc = ctx.createOscillator();
                osc.connect(ctx.destination);
                osc.frequency.value = 1000;
                osc.start(ctx.currentTime + i * 0.5);
                osc.stop(ctx.currentTime + i * 0.5 + 0.3);
            }
        """))
    else:
        display(Javascript("""
            var ctx = new AudioContext();
            var osc = ctx.createOscillator();
            osc.connect(ctx.destination);
            osc.frequency.value = 600;
            osc.start();
            osc.stop(ctx.currentTime + 0.5);
        """))

# ============================================================
# JavaScript — بيفتح الكاميرا في المتصفح
# ============================================================
CAPTURE_JS = """
async function captureFrame() {
    const stream = await navigator.mediaDevices.getUserMedia({video: true});
    const video  = document.createElement('video');
    video.srcObject = stream;
    await video.play();

    const canvas = document.createElement('canvas');
    canvas.width  = video.videoWidth;
    canvas.height = video.videoHeight;
    canvas.getContext('2d').drawImage(video, 0, 0);

    stream.getTracks().forEach(t => t.stop());
    return canvas.toDataURL('image/jpeg', 0.8);
}
captureFrame();
"""

# ============================================================
# Main Test Loop
# ============================================================
def run_camera_test():
    loaded_model = keras.models.load_model('saved_models/best_model.keras')
    print(f'Model loaded. بيصور لمدة {TEST_DURATION} ثانية...')
    print('=' * 45)

    start_time   = time.time()
    total_frames = 0
    cheat_frames = 0

    while True:
        elapsed   = time.time() - start_time
        remaining = TEST_DURATION - elapsed

        if elapsed >= TEST_DURATION:
            break

        # ── صور frame من الكاميرا ──────────────────────
        if USE_MOBILE_CAM:
            # كاميرا الموبايل عن طريق IP
            cap = cv2.VideoCapture(MOBILE_CAM_URL)
            ret, frame = cap.read()
            cap.release()
            if not ret:
                print('تحقق من الـ IP وإن الموبايل والكمبيوتر على نفس الـ WiFi')
                break
        else:
            # كاميرا اللاب عن طريق المتصفح
            b64 = eval_js(CAPTURE_JS)
            frame = base64_to_frame(b64)
            if frame is None:
                print('مش قادر يصور — تأكد إنك سمحت للمتصفح يوصل للكاميرا')
                break

        # ── Predict ────────────────────────────────────
        label, prob = predict_frame(frame, loaded_model)
        total_frames += 1
        if label == 'cheating':
            cheat_frames += 1

        # ── اطبع الحالة الحالية ────────────────────────
        clear_output(wait=True)
        icon = '⚠️ CHEATING' if label == 'cheating' else '✅ NORMAL'
        print(f'الوقت المتبقي : {remaining:.0f} ثانية')
        print(f'الحالة الحالية: {icon}  ({prob:.0%})')
        print(f'Frames analyzed: {total_frames} | Cheating: {cheat_frames}')
        print('=' * 45)

        time.sleep(CAPTURE_EVERY)

    # ── النتيجة النهائية ────────────────────────────────
    if total_frames == 0:
        print('لم يتم التقاط أي frames!')
        return

    cheat_ratio  = cheat_frames / total_frames
    final_result = 'CHEATING' if cheat_ratio >= CHEAT_THRESHOLD else 'NORMAL'

    clear_output(wait=True)
    print(f'Frames analyzed : {total_frames}')
    print(f'Cheating frames : {cheat_frames} ({cheat_ratio:.0%})')
    send_notification(final_result, cheat_ratio)


run_camera_test()